<a href="https://colab.research.google.com/github/Orefle2003/AnswerTime-MetricNLP/blob/model-experiments-1/bertopic-individual_yelp_review_analysis_v0.8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install required libraries
!pip install datasets pandas

# Import necessary libraries
import pandas as pd
from datasets import load_dataset

# Load a sample of the Yelp reviews dataset
print("Loading the Yelp Polarity dataset...")
dataset = load_dataset("yelp_polarity", split="test[:200]")  # Load a subset of 200 reviews

# Convert the dataset to a Pandas DataFrame
reviews = pd.DataFrame(dataset)

# Randomly select 100 reviews
sampled_reviews = reviews.sample(100, random_state=42)  # Ensure reproducibility

# Print the contents of 100 reviews
print("\n--- Printing 100 Yelp Reviews ---\n")
for i, review in enumerate(sampled_reviews['text']):
    print(f"Review {i + 1}:\n{review}\n{'-' * 80}\n")


Loading the Yelp Polarity dataset...

--- Printing 100 Yelp Reviews ---

Review 1:
a classic. nothing else to say
--------------------------------------------------------------------------------

Review 2:
Very bad purchase experience. I bought a shirt with a hole covered in the rolled up sleeves, but they denied my request to return it. I am so angery at this and will never shop their chothes anymore.
--------------------------------------------------------------------------------

Review 3:
I've been here quite a few times, and I always try to like it cause theres a lack of good sports bars near Pitt. They have a ton of nice HDTVs.\n\nEverytime I go, there is always a weird taste to one (or more) of the beers that I get. I was just there for the Pitt/Oakland game and got a Redhook. I drink this beer regularly, and something was wrong with the beer. Its definitely hard to describe, but something is always off (others in my party notice it too).\n\nThe food is hit or miss. I got the Fr

In [5]:
!pip install datasets pandas bertopic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 3.4 MB/s eta 0:00:00


In [13]:
!pip install vaderSentiment
!pip install bertopic vaderSentiment scikit-learn


In [15]:
from bertopic import BERTopic
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

def analyze_reviews_with_topics_and_mapping(texts):
    # Step 1: Extract topics using BERTopic with stopword filtering
    vectorizer = CountVectorizer(stop_words="english")  # Filter stopwords
    topic_model = BERTopic(vectorizer_model=vectorizer)
    topics, probs = topic_model.fit_transform(texts)
    topic_info = topic_model.get_topic_info()

    # Step 2: Auto-generate topic names
    auto_topic_names = {}
    for topic_id in topic_info["Topic"]:
        if topic_id == -1:
            auto_topic_names[topic_id] = "Outliers"
        else:
            # Extract the top 3 representative words for each topic
            words = topic_model.get_topic(topic_id)[:3]
            topic_name = ", ".join([word[0] for word in words])
            auto_topic_names[topic_id] = topic_name

    # Apply the auto-generated topic names
    topic_model.set_topic_labels(auto_topic_names)

    # Step 3: Perform sentiment analysis and map reviews
    analyzer = SentimentIntensityAnalyzer()
    topic_sentiments = {topic: {"positive": [], "negative": [], "neutral": []} for topic in topic_info["Topic"]}
    review_mappings = []  # To store reviews and their classifications

    for idx, (text, topic) in enumerate(zip(texts, topics)):
        sentiment_score = analyzer.polarity_scores(text)
        if sentiment_score['compound'] > 0.05:
            sentiment = "positive"
        elif sentiment_score['compound'] < -0.05:
            sentiment = "negative"
        else:
            sentiment = "neutral"

        # Map the review to the sentiment and topic
        topic_name = auto_topic_names.get(topic, f"Topic {topic}")
        topic_sentiments[topic][sentiment].append(f"Review {idx + 1}")
        review_mappings.append((f"Review {idx + 1}", text, topic_name, sentiment))

    # Step 4: Calculate percentages
    total_reviews = len(texts)
    stats = {}
    for topic, sentiments in topic_sentiments.items():
        stats[auto_topic_names.get(topic, f"Topic {topic}")] = {
            "positive": round((len(sentiments["positive"]) / total_reviews) * 100, 2),
            "negative": round((len(sentiments["negative"]) / total_reviews) * 100, 2),
            "neutral": round((len(sentiments["neutral"]) / total_reviews) * 100, 2),
        }

    return stats, review_mappings, topic_sentiments

# Example usage
try:
    # Assuming `texts` contains the reviews
    stats, review_mappings, topic_sentiments = analyze_reviews_with_topics_and_mapping(texts)

    # Displaying reviews in a readable format
    print("\n--- Review Mappings ---\n")
    for review_id, text, topic, sentiment in review_mappings:
        print(f"{review_id}:\n{text}\nTopic: {topic}\nSentiment: {sentiment}\n{'-' * 80}")

    # Displaying the sentiment insights per topic with example reviews
    print("\n--- Sentiment Insights with Examples ---\n")
    for topic, sentiments in topic_sentiments.items():
        print(f"{topic}:\n"
              f"  Positive: {len(sentiments['positive'])} ({', '.join(sentiments['positive'])})\n"
              f"  Negative: {len(sentiments['negative'])} ({', '.join(sentiments['negative'])})\n"
              f"  Neutral: {len(sentiments['neutral'])} ({', '.join(sentiments['neutral'])})\n")

except NameError:
    print("It seems the required dataset or packages are not properly loaded. Run this on a local setup.")



--- Review Mappings ---

Review 1:
a classic. nothing else to say
Topic: place, food, like
Sentiment: neutral
--------------------------------------------------------------------------------
Review 2:
Very bad purchase experience. I bought a shirt with a hole covered in the rolled up sleeves, but they denied my request to return it. I am so angery at this and will never shop their chothes anymore.
Topic: Outliers
Sentiment: negative
--------------------------------------------------------------------------------
Review 3:
I've been here quite a few times, and I always try to like it cause theres a lack of good sports bars near Pitt. They have a ton of nice HDTVs.\n\nEverytime I go, there is always a weird taste to one (or more) of the beers that I get. I was just there for the Pitt/Oakland game and got a Redhook. I drink this beer regularly, and something was wrong with the beer. Its definitely hard to describe, but something is always off (others in my party notice it too).\n\nThe fo